# StarDist H5 Dataset Visualizer
Interactive visualization of paired raw / instance-label patches from the unified SmartPatches H5.

Reads `/train/raw` + `/train/label`. The label image is the instance-label int32 array StarDist's dataset uses to derive `(object_prob, ray_distances)` targets on the fly. Each label ID gets a distinct random colour; background is black.

Patches are stored already whole-volume percentile-normalised (CARE-style), so `raw` should land in roughly `[0, 1]`. If you see big-integer ranges, the H5 was generated with the old code path — regenerate before training.

In [3]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ipywidgets import IntSlider, Dropdown, VBox, Output, Label
import ipywidgets as widgets
from IPython.display import display, clear_output

In [4]:
# Set the path to your SmartPatches H5 file.
H5_PATH = "/mnt/jean-zay/segmentation_training/xenopus_segmentation.h5"

In [5]:
# Open H5 file and inspect structure
h5f = h5py.File(H5_PATH, 'r')

print("H5 File Structure:")
print(f"Root keys: {list(h5f.keys())}")
for split in h5f.keys():
    print(f"\n{split}/:")
    for key in h5f[split].keys():
        ds = h5f[split][key]
        sample = ds[0]
        print(f"  {key}: {ds.shape}, dtype={ds.dtype}, "
              f"min={sample.min():.4f}, max={sample.max():.4f}")

H5 File Structure:
Root keys: ['train', 'val']

train/:
  label: (35, 16, 256, 256), dtype=int32, min=0.0000, max=308.0000
  mask: (35, 16, 256, 256), dtype=uint8, min=0.0000, max=1.0000
  raw: (35, 16, 256, 256), dtype=float32, min=0.0000, max=1.0000

val/:
  label: (46, 16, 256, 256), dtype=int32, min=0.0000, max=179.0000
  mask: (46, 16, 256, 256), dtype=uint8, min=0.0000, max=1.0000
  raw: (46, 16, 256, 256), dtype=float32, min=0.0000, max=1.0000


In [6]:
def random_label_cmap(n=2048):
    """Distinct random colour per integer label; 0 → black for background."""
    rng = np.random.default_rng(42)
    colors = rng.uniform(0.2, 1.0, size=(n, 3))
    colors[0] = [0, 0, 0]
    return ListedColormap(colors)

LABEL_CMAP = random_label_cmap()

In [7]:
class StarDistH5Visualizer:
    """3D-aware raw / instance-label viewer for the unified SmartPatches H5.

    Handles both 2D and 3D patches transparently — the Z slider is
    hidden for 2D data. Labels are coloured per instance ID using a
    fixed-seed random colormap so the same ID gets the same colour
    across redraws.
    """

    def __init__(self, h5_file):
        self.h5f = h5_file
        self.current_split = 'train'
        self._bind_split(self.current_split)

        self.output = Output()
        self.stats_output = Output()

    def _bind_split(self, split):
        grp = self.h5f[split]
        self.raw = grp['raw']
        self.label = grp['label']
        self.n_samples = self.raw.shape[0]
        self.is_3d = self.raw.ndim == 4  # (N, Z, Y, X) for 3D, (N, Y, X) for 2D
        self.n_z = self.raw.shape[1] if self.is_3d else 1

    def switch_split(self, split):
        self.current_split = split
        self._bind_split(split)

    def visualize(self, sample_idx, z_idx, show_overlay):
        with self.output:
            clear_output(wait=True)

            raw_patch = self.raw[sample_idx]
            label_patch = self.label[sample_idx]

            if self.is_3d:
                raw_slice = raw_patch[z_idx]
                label_slice = label_patch[z_idx]
                z_label = f", Z={z_idx}"
            else:
                raw_slice = raw_patch
                label_slice = label_patch
                z_label = ""

            ncols = 3 if show_overlay else 2
            fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))

            axes[0].imshow(raw_slice, cmap='gray')
            axes[0].set_title(f'Raw — Sample {sample_idx}{z_label}')
            axes[0].axis('off')

            n_labels = int((np.unique(label_slice) > 0).sum())
            axes[1].imshow(
                label_slice.astype(int) % LABEL_CMAP.N,
                cmap=LABEL_CMAP, interpolation='nearest'
            )
            axes[1].set_title(f'Instance labels — {n_labels} IDs in slice')
            axes[1].axis('off')

            if show_overlay:
                axes[2].imshow(raw_slice, cmap='gray')
                rgba = LABEL_CMAP(label_slice.astype(int) % LABEL_CMAP.N)
                rgba[..., 3] = np.where(label_slice > 0, 0.45, 0.0)
                axes[2].imshow(rgba)
                axes[2].set_title('Overlay (raw + labels)')
                axes[2].axis('off')

            plt.tight_layout()
            plt.show()

        with self.stats_output:
            clear_output(wait=True)
            print("=" * 60)
            print(f"Patch {sample_idx} — {self.current_split} split")
            print("=" * 60)
            print(f"Raw   shape: {raw_patch.shape}, dtype: {raw_patch.dtype}")
            print(f"Raw   — min: {raw_patch.min():.4f}, max: {raw_patch.max():.4f}, "
                  f"mean: {raw_patch.mean():.4f}, std: {raw_patch.std():.4f}")
            print(f"Label shape: {label_patch.shape}, dtype: {label_patch.dtype}")
            ids = np.unique(label_patch)
            print(f"Label — {len(ids)-1 if 0 in ids else len(ids)} instance IDs in patch "
                  f"(min={int(ids.min())}, max={int(ids.max())})")
            fg_frac = float((label_patch > 0).mean()) * 100
            print(f"Foreground voxels: {fg_frac:.2f}% of patch")
            print("=" * 60)

    def create_widgets(self):
        split_dropdown = Dropdown(
            options=list(self.h5f.keys()),
            value=self.current_split,
            description='Split:'
        )

        sample_slider = IntSlider(
            min=0, max=self.n_samples - 1, step=1, value=0,
            description='Sample:', continuous_update=False
        )

        z_slider = IntSlider(
            min=0, max=max(self.n_z - 1, 0), step=1, value=self.n_z // 2,
            description='Z slice:', continuous_update=False
        )
        if not self.is_3d:
            z_slider.layout.display = 'none'

        overlay_checkbox = widgets.Checkbox(value=True, description='Show Overlay')

        def on_split_change(change):
            self.switch_split(change['new'])
            sample_slider.max = self.n_samples - 1
            sample_slider.value = min(sample_slider.value, self.n_samples - 1)
            z_slider.max = max(self.n_z - 1, 0)
            z_slider.value = min(z_slider.value, max(self.n_z - 1, 0))
            z_slider.layout.display = 'flex' if self.is_3d else 'none'
            self.visualize(sample_slider.value, z_slider.value, overlay_checkbox.value)

        split_dropdown.observe(on_split_change, names='value')

        def update(sample_idx, z_idx, show_overlay):
            self.visualize(sample_idx, z_idx, show_overlay)

        widgets.interactive(
            update,
            sample_idx=sample_slider,
            z_idx=z_slider,
            show_overlay=overlay_checkbox,
        )

        controls = VBox([
            Label(f'Dataset: {self.n_samples} patches, shape: {self.raw.shape}'),
            split_dropdown,
            sample_slider,
            z_slider,
            overlay_checkbox,
        ])

        display(controls)
        display(self.output)
        display(self.stats_output)

        self.visualize(0, self.n_z // 2, True)

In [8]:
viz = StarDistH5Visualizer(h5f)
viz.create_widgets()

Output()

Output()